# FIM Line Model — Byte-Level BPE + RoPE, trained from scratch

The same encoder-decoder as `L-bpe-rope` (`modules/models/L_rope_model.py`, unchanged), only the data differ:

* encoder: `<BOS> prefix <FIM_HOLE> suffix <EOS>` — the bidirectional encoder reads the code on **both** sides of the cursor;
* decoder: `<BOS> middle <EOS>` — the decoder starts from `<BOS>`, exactly as `LineModel.generate` does at inference.

`<FIM_HOLE>` is added to the existing BPE vocabulary; the result is saved as `tokenizer_bpe_fim.json`, `tokenizer_bpe.json` is not touched.

In [ ]:
%tb
import os, math, random, glob
from pathlib import Path
from typing import Optional

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler
from tqdm import tqdm
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

import warnings
warnings.filterwarnings("ignore")

from modules.Plotting import MetricLog, plot_metrics
from modules.early_stopping import EarlyStopping
from modules.hand_testing import hand_test_fim_repl
from modules.best_model_saver import BestModelSaver
from modules.datasets.loading import load_files
from modules.datasets.line_dataset import collate_line
from modules.datasets.fim_dataset import FIMLineDataset
from modules.tokenizers.BPE_tokenizer import BPECodeTokenizer
from modules.models.L_rope_model import LineModel, ModelCfg
from modules.fim import add_fim_token
from modules.evaluation_fim import evaluate_fim_model

WORKDIR = r'C:\Users\Roman\Documents\Projects\code_autocomplete'
# WORKDIR = r'/home/ubuntu/code_autocomplete/'
print(f"WORKDIR: {WORKDIR}")
LINE_MODEL_NAME = "line_model_FIM_bpe_rope"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Device] {device}")

## Training loop

In [ ]:
def _clip_norm(model, max_norm=1.0):
    return nn.utils.clip_grad_norm_(model.parameters(), max_norm).item()


def _seq2seq_loss(model, crit, src, tgt, pad_id):
    # tgt = <BOS> middle <EOS>: the decoder reads <BOS> middle and predicts middle <EOS>
    dec_in, dec_out = tgt[:, :-1], tgt[:, 1:]
    logits = model(src, dec_in, src_key_padding_mask=(src == pad_id),
                   tgt_key_padding_mask=(dec_in == pad_id))
    return crit(logits.reshape(-1, logits.size(-1)), dec_out.reshape(-1)), logits, dec_out


def train_line_model(model, train_dl, val_dl, epochs, lr, device,
                     saver, log, plot_dir, pad_id=0,
                     label_smoothing=0.1, warmup_frac=0.05,
                     patience=3, use_amp=True):
    tqdm.write(f"[Line] DataLoader — {len(train_dl)} train batches, {len(val_dl)} val batches")
    opt = AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    total_steps = len(train_dl) * epochs
    warmup_steps = max(1, int(total_steps * warmup_frac))
    sched = get_cosine_schedule_with_warmup(opt, warmup_steps, total_steps)
    crit = nn.CrossEntropyLoss(ignore_index=pad_id, label_smoothing=label_smoothing)
    amp_enabled = use_amp and device.type == "cuda"
    scaler = GradScaler("cuda", enabled=amp_enabled)
    stopper = EarlyStopping(patience=patience)

    for ep in range(1, epochs + 1):
        model.train()
        t_loss = t_steps = 0; gn = 0.0
        for src, tgt in tqdm(train_dl, desc=f"[Line] Epoch {ep}/{epochs} train", leave=False, unit="batch"):
            src, tgt = src.to(device, non_blocking=True), tgt.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with autocast("cuda", dtype=torch.float16, enabled=amp_enabled):
                loss, _, _ = _seq2seq_loss(model, crit, src, tgt, pad_id)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            gn = _clip_norm(model)
            scaler.step(opt); scaler.update(); sched.step()
            t_loss += loss.item()
            t_steps += 1
        tl = t_loss / t_steps

        model.eval()
        v_loss = v_steps = v_acc = 0
        with torch.no_grad():
            for src, tgt in tqdm(val_dl, desc=f"[Line] Epoch {ep}/{epochs} val  ", leave=False, unit="batch"):
                src, tgt = src.to(device, non_blocking=True), tgt.to(device, non_blocking=True)
                with autocast("cuda", dtype=torch.float16, enabled=amp_enabled):
                    loss, logits, dec_out = _seq2seq_loss(model, crit, src, tgt, pad_id)
                mask = (dec_out != pad_id)
                if mask.any():
                    v_acc += (logits.argmax(-1)[mask] == dec_out[mask]).float().mean().item()
                v_loss += loss.item()
                v_steps += 1
        vl = v_loss / v_steps if v_steps else tl
        va = v_acc / v_steps if v_steps else 0.0

        log.append(train_loss=tl, val_loss=vl,
                   train_ppl=math.exp(min(tl, 20)), val_ppl=math.exp(min(vl, 20)),
                   lr=opt.param_groups[0]["lr"], token_acc=va, grad_norm=gn)
        tqdm.write(f"[Line  ep {ep:3d}] train_loss={tl:.4f}  val_loss={vl:.4f}"
                   f"  ppl={math.exp(min(vl,20)):.1f}  val_acc={va:.3f}"
                   f"  lr={opt.param_groups[0]['lr']:.2e}")
        saver.save(model, vl, ep)
        if ep % max(1, epochs // 5) == 0 or ep == epochs:
            plot_metrics(log, f"{LINE_MODEL_NAME.replace('_', ' ')} — Epoch {ep}",
                         f"{plot_dir}/{LINE_MODEL_NAME}_ep{ep:02d}.png")
        if stopper(vl):
            tqdm.write(f"[Early stop] val_loss did not improve for {stopper.patience} epochs — stopping.")
            break

    plot_metrics(log, f"{LINE_MODEL_NAME.replace('_', ' ')} — Final",
                 f"{plot_dir}/{LINE_MODEL_NAME}_final.png")

## Main

In [ ]:
class Arguments():
    def __init__(self,
                 data_dir=f"{WORKDIR}/Clean_Dataset",   # train/ val/ test/ written by DataPreparation
                 ckpt_dir=f"{WORKDIR}/checkpoints/{LINE_MODEL_NAME}",
                 plot_dir=f"{WORKDIR}/plots/{LINE_MODEL_NAME}",
                 results_dir=f"{WORKDIR}/results/{LINE_MODEL_NAME}",
                 base_tokenizer="tokenizer_bpe.json",    # BPE merges shared with T-/L-bpe-rope
                 tokenizer="tokenizer_bpe_fim.json",     # + <FIM_HOLE>
                 epochs=5, batch=32, lr=5e-4, vocab_size=16000, max_files=0,
                 d_model=256, n_layers=4, n_heads=8,
                 max_prefix=256, max_suffix=128, max_middle=48,
                 line_sample_rate=1.0,
                 label_smoothing=0.1, warmup_frac=0.05, patience=3, use_amp=True,
                 seed=42, test=False):
        for k, v in locals().items():
            if k != "self": setattr(self, k, v)


def best_ckpt(ckpt_dir: str) -> Optional[str]:
    """Lowest val-loss checkpoint (file names sort by epoch, not by loss)."""
    paths = glob.glob(f"{ckpt_dir}/{LINE_MODEL_NAME}_*.pt")
    return min(paths, key=lambda p: float(p.rsplit("loss", 1)[1][:-3])) if paths else None


def get_tokenizer(args) -> BPECodeTokenizer:
    if os.path.exists(args.tokenizer):
        return BPECodeTokenizer.load(args.tokenizer)
    if os.path.exists(args.base_tokenizer):
        print(f"[Tokenizer] loading {args.base_tokenizer}")
        tokenizer = BPECodeTokenizer.load(args.base_tokenizer)
    else:
        print("[Tokenizer] training byte-level BPE from data …")
        tokenizer = BPECodeTokenizer(vocab_size=args.vocab_size)
        tokenizer.build(load_files(f"{args.data_dir}/train", args.max_files))
    print(f"[Tokenizer] <FIM_HOLE> id = {add_fim_token(tokenizer)}")
    tokenizer.save(args.tokenizer)
    return tokenizer


def build_model(args, tokenizer) -> LineModel:
    cfg = ModelCfg(vocab=tokenizer.vocab, d_model=args.d_model, n_heads=args.n_heads,
                   n_layers=args.n_layers, d_ff=args.d_model * 4,
                   max_len=args.max_prefix + args.max_suffix + 8)
    return LineModel(cfg).to(device)


def main(args):
    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(args.seed)
    os.makedirs(args.ckpt_dir, exist_ok=True)
    os.makedirs(args.plot_dir, exist_ok=True)
    torch.serialization.add_safe_globals([ModelCfg])

    tokenizer = get_tokenizer(args)
    model = build_model(args, tokenizer)

    if args.test:
        path = best_ckpt(args.ckpt_dir)
        if path:
            model.load_state_dict(torch.load(path, map_location=device, weights_only=False)["model_state"])
            print(f"[Loaded] line model from {path}")
        hand_test_fim_repl(model, tokenizer, device, kind="custom")
        return

    print("[Loading] Started loading")
    tr_txt = load_files(f"{args.data_dir}/train", args.max_files)
    va_txt = load_files(f"{args.data_dir}/val", args.max_files)
    if not tr_txt or not va_txt:
        print("[ERROR] no data — expected Clean_Dataset/train and Clean_Dataset/val")
        return
    print("[Loading] Ended loading")

    ds_kw = dict(max_prefix=args.max_prefix, max_suffix=args.max_suffix,
                 max_middle=args.max_middle, seed=args.seed)
    tr_ds = FIMLineDataset(tr_txt, tokenizer, line_sample_rate=args.line_sample_rate, **ds_kw)
    va_ds = FIMLineDataset(va_txt, tokenizer, deterministic=True, **ds_kw)
    collate = lambda b: collate_line(b, tokenizer.pad_id)
    tr_dl = DataLoader(tr_ds, args.batch, shuffle=True, collate_fn=collate, num_workers=0, pin_memory=True)
    va_dl = DataLoader(va_ds, args.batch, shuffle=False, collate_fn=collate, num_workers=0, pin_memory=True)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[Line  Model] {n_params/1e6:.2f}M parameters (FIM, BPE+RoPE)")

    saver = BestModelSaver(args.ckpt_dir, LINE_MODEL_NAME)
    train_line_model(model, tr_dl, va_dl, args.epochs, args.lr, device, saver, MetricLog(), args.plot_dir,
                     pad_id=tokenizer.pad_id, label_smoothing=args.label_smoothing,
                     warmup_frac=args.warmup_frac, patience=args.patience, use_amp=args.use_amp)


main(Arguments())
# main(Arguments(test=True))

## Evaluation

On `Clean_Dataset/test`, with and without the suffix (ablation).

In [ ]:
def evaluate(args, n_samples=500):
    tokenizer = get_tokenizer(args)
    model = build_model(args, tokenizer)
    path = best_ckpt(args.ckpt_dir)
    assert path, f"no checkpoint in {args.ckpt_dir}"
    model.load_state_dict(torch.load(path, map_location=device, weights_only=False)["model_state"])
    test_txt = load_files(f"{args.data_dir}/test", 200)
    kw = dict(kind="custom", n_samples=n_samples, out_dir=args.results_dir, title="FIM Line - BPE + RoPE",
              max_prefix=args.max_prefix, max_suffix=args.max_suffix)

    results = {"trained": evaluate_fim_model(model, tokenizer, test_txt, device, **kw),
               "no suffix": evaluate_fim_model(model, tokenizer, test_txt, device,
                                               use_suffix=False, tag="NO-SUFFIX ", **kw)}
    for name, r in results.items():
        print(f"  {name:10s} exact={r['exact_match']:.3f}  edit={r['edit_similarity']:.3f}  "
              f"syntax={r['syntax_valid']:.3f}  first_tok={r['first_token_match']:.3f}  "
              f"p50={r['latency_p50']:.0f} ms")
    return results


fim_results = evaluate(Arguments())